# 07 · DeepLabV3+ vs SegFormer, ensemble test, and final model choice

No training here. The notebook reads the outputs of notebooks 05 and 06 and runs three analyses on the local test data:

1. **DeepLabV3+ R101 vs SegFormer MiT-B3**, for A (zero-shot) and B (adapted):
   - mean ± std over the 3 seeds, with the tie rule from the study design: a **tie** when |difference| < the larger std;
   - per-image paired bootstrap (2,000 resamples) of Δ = SegFormer − DeepLab; significant when the 95% CI excludes 0;
   - number of runs (seed × configuration) won by each model.
2. **Ensemble test** (seed 44, 6 configurations): the two B models' probabilities are averaged with weight w, p = w·SegFormer + (1−w)·DeepLab, w ∈ {0, 0.25, 0.5, 0.75, 1}.
   - The weight is chosen on the **partner configuration** (c0↔c1, c2↔c3, c4↔c5, whose test sets are disjoint) and scored on the configuration itself, so test data is never used to choose it.
   - A plain average (w = 0.5, no selection) is reported as well.
   - Ensembles are compared with the best single model by paired bootstrap.
3. **Final model choice** (model F) for deployment, based on 1 and 2 and on inference time. A difference counts only if it is statistically significant **and** at least `MIN_PRACTICAL_DIFF` (1 percentage point) in mIoU litter.

Metrics follow notebooks 05 and 06: per-image confusion matrices at short side 512, ignored pixels (255) removed, **mIoU litter** = mean IoU of artificial and natural.

**Kaggle inputs:** the outputs of notebooks 05 (`adaptation_deeplab_r101`) and 06 (`adaptation_segformer_b3`), found automatically, and the `beach-litter` dataset (local masks for the ensemble test). No GPU needed.

## 1. Configuration

In [ ]:
import json
import re
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/comparison_ensemble")
OUT.mkdir(parents=True, exist_ok=True)

DL_NAME, SF_NAME = "adaptation_deeplab_r101", "adaptation_segformer_b3"
DL_DIR = None
SF_DIR = None

LOCAL_DIR = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/data_lokal_cirebonbali/DATA LOKAL")
MASK_NAME_FIX = {"TT10": "TT110"}
EXPECTED_LOCAL = {"bali": 61, "cirebon": 89}
SHORT_SIDE = 512
IGNORE = 255

PROBA_SEED = 44
WEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)
PARTNER = {0: 1, 1: 0, 2: 3, 3: 2, 4: 5, 5: 4}
N_BOOT = 2000
BOOT_SEED = 123
MIN_PRACTICAL_DIFF = 0.01
ARCH = {"DL": "DeepLabV3+ R101", "SF": "SegFormer MiT-B3"}

CM_COLS = [f"cm_{g}{p}" for g in range(3) for p in range(3)]
TAGS = {1: "art", 2: "nat"}


def find_output(name: str, configured):
    if configured is not None:
        return Path(configured)
    found = sorted({p.parent.parent for p in INPUT_ROOT.rglob("runs/*.json") if p.parent.parent.name == name})
    assert len(found) == 1, f"Folder {name}: found {len(found)} ({found}). Set the path manually."
    return found[0]


DL_DIR, SF_DIR = find_output(DL_NAME, DL_DIR), find_output(SF_NAME, SF_DIR)
print("DeepLab  :", DL_DIR)
print("SegFormer:", SF_DIR)

if not LOCAL_DIR.is_dir():
    hits = [p for p in INPUT_ROOT.rglob("DATA LOKAL") if (p / "images").is_dir() and (p / "mask").is_dir()]
    assert len(hits) == 1, f"DATA LOKAL folder found {len(hits)} times: {hits}. Set LOCAL_DIR manually."
    LOCAL_DIR = hits[0]
print("Local data:", LOCAL_DIR)

## 2. Load both models and check the design matches
Both notebooks must use the same local splits, have finished all 21 runs, and have the seed-44 probabilities.

In [ ]:
splits = {}
for key, d in (("DL", DL_DIR), ("SF", SF_DIR)):
    splits[key] = json.loads((d / "splits" / "local_configs.json").read_text())
assert splits["DL"] == splits["SF"], "The two notebooks use different local splits; the comparison is invalid."
LOCAL_CONFIGS = {int(k): v for k, v in splits["DL"].items()}


def load_eval(d: Path, arch: str) -> pd.DataFrame:
    df = pd.concat([pd.read_csv(f) for f in sorted((d / "eval").glob("*.csv"))], ignore_index=True)
    return df.assign(arch=arch)


ev = pd.concat([load_eval(DL_DIR, "DL"), load_eval(SF_DIR, "SF")], ignore_index=True)
SOURCE_OF = dict(zip(ev.id, ev.source))
runs = ev.drop_duplicates(["arch", "kind", "seed", "config"]).groupby(["arch", "kind"]).size().unstack()
print("Runs per model:")
print(runs)
seeds = sorted(set(ev.seed))
for arch in ("DL", "SF"):
    for kind, n in (("A", len(seeds)), ("B", len(seeds) * len(LOCAL_CONFIGS))):
        assert runs.loc[arch, kind] == n, f"{ARCH[arch]} model {kind}: {runs.loc[arch, kind]} runs, expected {n}"
for d in (DL_DIR, SF_DIR):
    missing = [c for c in LOCAL_CONFIGS if not (d / "proba" / f"B_s{PROBA_SEED}_c{c}.npz").is_file()]
    assert not missing, f"{d.name}: seed-{PROBA_SEED} probabilities missing for configurations {missing}"
print(f"Seeds: {seeds} | configurations: {sorted(LOCAL_CONFIGS)} | seed-{PROBA_SEED} probabilities: complete")

ms = ev[ev.kind == "A"].groupby("arch").ms.median()
print("Median inference time (ms/image):", ms.round(1).to_dict())

## 3. Metrics and bootstrap
Every metric comes from per-image confusion matrices. IoU, precision, recall and Dice are pooled over all pixels; area MAE is the mean absolute difference in per-image area percentage.

In [ ]:
def counts(frame: pd.DataFrame, c: int):
    tp = frame[f"cm_{c}{c}"].to_numpy(float)
    gt = sum(frame[f"cm_{c}{p}"].to_numpy(float) for p in range(3))
    pr = sum(frame[f"cm_{g}{c}"].to_numpy(float) for g in range(3))
    return tp, gt, pr


def area_pct(frame: pd.DataFrame):
    valid = sum(frame[c].to_numpy(float) for c in CM_COLS)
    out = {}
    for c, tag in TAGS.items():
        _, gt, pr = counts(frame, c)
        out[f"gt_{tag}"], out[f"pr_{tag}"] = 100 * gt / valid, 100 * pr / valid
    out["gt_total"], out["pr_total"] = out["gt_art"] + out["gt_nat"], out["pr_art"] + out["pr_nat"]
    return out


def metric_arrays(frame: pd.DataFrame) -> dict:
    arr = {}
    for c, tag in TAGS.items():
        tp, gt, pr = counts(frame, c)
        arr[f"tp_{tag}"], arr[f"gt_{tag}"], arr[f"pr_{tag}"] = tp, gt, pr
    a = area_pct(frame)
    for tag in ("art", "nat", "total"):
        arr[f"ae_{tag}"] = np.abs(a[f"pr_{tag}"] - a[f"gt_{tag}"])
    return arr


def compute(arr: dict, ix=None) -> dict:
    def s(key):
        v = arr[key]
        return v.sum() if ix is None else v[ix].sum(1)

    def m(key):
        v = arr[key]
        return v.mean() if ix is None else v[ix].mean(1)

    out = {}
    with np.errstate(all="ignore"):
        for tag in TAGS.values():
            tp, gt, pr = s(f"tp_{tag}"), s(f"gt_{tag}"), s(f"pr_{tag}")
            out[f"IoU_{tag}"] = tp / (gt + pr - tp)
            out[f"Dice_{tag}"] = 2 * tp / (gt + pr)
            out[f"precision_{tag}"] = tp / pr
            out[f"recall_{tag}"] = tp / gt
        out["mIoU_litter"] = (out["IoU_art"] + out["IoU_nat"]) / 2
        for tag in ("art", "nat", "total"):
            out[f"area_MAE_{tag}_pp"] = m(f"ae_{tag}")
    return out


MAIN_METRICS = ["mIoU_litter", "IoU_art", "IoU_nat", "Dice_art", "Dice_nat",
                "area_MAE_art_pp", "area_MAE_nat_pp", "area_MAE_total_pp"]
LOWER_IS_BETTER = {m for m in MAIN_METRICS if m.startswith("MAE")}


def paired_bootstrap(f1: pd.DataFrame, f2: pd.DataFrame, label1: str, label2: str) -> pd.DataFrame:
    assert (f1.id.to_numpy() == f2.id.to_numpy()).all()
    rng = np.random.default_rng(BOOT_SEED)
    rows = []
    groups = [("all", np.arange(len(f1)))] + [(s, np.flatnonzero(f1.source.to_numpy() == s))
                                              for s in sorted(f1.source.unique())]
    for name, idx in groups:
        a1, a2 = metric_arrays(f1.iloc[idx]), metric_arrays(f2.iloc[idx])
        ix = rng.integers(0, len(idx), size=(N_BOOT, len(idx)))
        v1, v2 = compute(a1), compute(a2)
        b1, b2 = compute(a1, ix), compute(a2, ix)
        for met in MAIN_METRICS:
            delta = b2[met] - b1[met]
            lo, hi = np.nanpercentile(delta, [2.5, 97.5])
            better = label2 if (v2[met] < v1[met]) == (met in LOWER_IS_BETTER) else label1
            better = better.replace("value_", "")
            rows.append({"group": name, "n_images": len(idx), "metric": met, label1: v1[met], label2: v2[met],
                         "delta": v2[met] - v1[met], "ci95_low": lo, "ci95_high": hi,
                         "significant": bool(lo > 0 or hi < 0),
                         "better": better if (lo > 0 or hi < 0) else "no significant difference"})
    return pd.DataFrame(rows)


def per_image(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.groupby("id")[CM_COLS].mean().reset_index()
    out["source"] = out.id.map(SOURCE_OF)
    return out.sort_values("id").reset_index(drop=True)

## 4. DeepLabV3+ vs SegFormer
### 4a. Mean ± std over seeds (tie rule)
For each seed, per-image confusion matrices are averaged over configurations (B) and metrics computed, giving one value per seed. The result is a **tie** when |mean SegFormer − mean DeepLab| is below the larger of the two stds.

In [ ]:
def model_frame(arch: str, kind: str) -> pd.DataFrame:
    if kind == "A":
        return ev[(ev.arch == arch) & (ev.kind == "A")]
    return ev[(ev.arch == arch) & (ev.kind == "B")]


seed_rows = []
for kind in ("A", "B"):
    for arch in ("DL", "SF"):
        for seed in seeds:
            f = per_image(model_frame(arch, kind).query("seed == @seed"))
            seed_rows.append({"model": kind, "arch": arch, "seed": seed, **compute(metric_arrays(f))})
per_seed = pd.DataFrame(seed_rows)
per_seed.to_csv(OUT / "metrics_per_seed.csv", index=False)

tie_rows = []
for kind in ("A", "B"):
    for met in MAIN_METRICS:
        g = per_seed[per_seed.model == kind].groupby("arch")[met]
        mean, std = g.mean(), g.std(ddof=1)
        diff = mean["SF"] - mean["DL"]
        tie = abs(diff) < max(std["DL"], std["SF"])
        if tie:
            verdict = "tie"
        else:
            verdict = ARCH["SF"] if (diff < 0) == (met in LOWER_IS_BETTER) else ARCH["DL"]
        tie_rows.append({"model": kind, "metric": met,
                         "DeepLab (mean ± std)": f"{mean['DL']:.4f} ± {std['DL']:.4f}",
                         "SegFormer (mean ± std)": f"{mean['SF']:.4f} ± {std['SF']:.4f}",
                         "diff SF−DL": diff, "max_std": max(std["DL"], std["SF"]), "verdict": verdict})
tie_table = pd.DataFrame(tie_rows)
tie_table.to_csv(OUT / "mean_std_per_seed.csv", index=False)
display(tie_table.round(4))

### 4b. Per-image paired bootstrap (Δ = SegFormer − DeepLab)
Each local image is represented by its confusion matrix averaged over all seeds (A) or all seeds × configurations in which it is a test image (B). Both models are tested on the same images, seeds and configurations, so images can be resampled in pairs.

In [ ]:
boot_arch = []
for kind in ("A", "B"):
    f_dl, f_sf = per_image(model_frame("DL", kind)), per_image(model_frame("SF", kind))
    res = paired_bootstrap(f_dl, f_sf, "DeepLab", "SegFormer").assign(model=kind)
    boot_arch.append(res)
boot_arch = pd.concat(boot_arch, ignore_index=True)
boot_arch.to_csv(OUT / "significance_DL_vs_SF.csv", index=False)
for kind, label in (("A", "Model A (zero-shot)"), ("B", "Model B (adapted)")):
    print(f"\n{label}: Δ = SegFormer − DeepLab, 95% paired-bootstrap CI")
    display(boot_arch[boot_arch.model == kind].drop(columns="model").round(4))

### 4c. Runs won
mIoU litter per run (pooled over that run's test images), compared on the same seed and configuration.

In [ ]:
run_rows = []
for (arch, kind, seed, cfg), g in ev.groupby(["arch", "kind", "seed", "config"]):
    run_rows.append({"arch": arch, "model": kind, "seed": seed, "config": cfg,
                     "mIoU_litter": compute(metric_arrays(g))["mIoU_litter"]})
per_run = pd.DataFrame(run_rows).pivot_table(index=["model", "seed", "config"], columns="arch",
                                            values="mIoU_litter").reset_index()
per_run["diff_SF_DL"] = per_run.SF - per_run.DL
per_run.to_csv(OUT / "mIoU_per_run_DL_vs_SF.csv", index=False)
for kind in ("A", "B"):
    sub = per_run[per_run.model == kind]
    print(f"Model {kind}: SegFormer wins {(sub.diff_SF_DL > 0).sum()} of {len(sub)} runs | "
          f"mean difference {sub.diff_SF_DL.mean():+.4f} (min {sub.diff_SF_DL.min():+.4f}, "
          f"max {sub.diff_SF_DL.max():+.4f})")

## 5. Ensemble test (seed 44)
### 5a. Local masks at the probability resolution
Masks are read with the same rules as notebooks 05 and 06 (RGBA: red → 1, green → 2, other bright colours → 255), matched to the image size, then resized to short side 512 with nearest-neighbour interpolation.

In [ ]:
def index_dir(directory: Path, strip_mask_suffix: bool) -> dict:
    index = {}
    for path in sorted(directory.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        stem = re.sub(r"_mask(?:\s*\d+)?$", "", path.stem, flags=re.I) if strip_mask_suffix else path.stem
        index[stem] = path
    return index


def read_local_mask(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.asarray(im if im.mode == "RGBA" else im.convert("RGB"))
    visible = arr[..., 3] >= 128 if arr.shape[2] == 4 else np.ones(arr.shape[:2], bool)
    rgb = arr[..., :3].astype(np.int16)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    red = visible & (r >= 128) & (r >= g + 25) & (r >= b + 25)
    green = visible & (g >= 128) & (g >= r + 25) & (g >= b + 25)
    ambiguous = visible & (rgb.max(axis=2) >= 128) & ~(red | green)
    labels = np.zeros(arr.shape[:2], np.uint8)
    labels[red], labels[green], labels[ambiguous] = 1, 2, IGNORE
    return labels


images, masks = index_dir(LOCAL_DIR / "images", False), index_dir(LOCAL_DIR / "mask", True)
for wrong, right in MASK_NAME_FIX.items():
    if wrong in masks and right not in masks:
        masks[right] = masks.pop(wrong)

GT = {}
for stem in tqdm(sorted(images), desc="Local masks"):
    with Image.open(images[stem]) as im:
        w, h = im.size
    label = read_local_mask(masks[stem])
    if label.shape != (h, w):
        label = cv2.resize(label, (w, h), interpolation=cv2.INTER_NEAREST)
    scale = SHORT_SIDE / min(h, w)
    GT["local_" + stem] = cv2.resize(label, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_NEAREST)
src_count = pd.Series([SOURCE_OF.get(i) for i in GT]).value_counts().to_dict()
assert src_count == EXPECTED_LOCAL, f"Unexpected local counts: {src_count}"
print(f"{len(GT)} local masks loaded: {src_count}")

### 5b. Confusion matrices for every weight
Probabilities are stored as uint8 (0–255) for artificial and natural; background = 255 minus both. As a check, w = 0 and w = 1 are compared with each model's original evaluation.

In [ ]:
def load_proba(d: Path, cfg: int) -> dict:
    with np.load(d / "proba" / f"B_s{PROBA_SEED}_c{cfg}.npz") as z:
        return {k: z[k] for k in z.files}


def to_full(p: np.ndarray) -> np.ndarray:
    p = p.astype(np.float32)
    return np.concatenate([np.clip(255 - p.sum(0, keepdims=True), 0, None), p], axis=0)


ens_rows = []
for cfg in tqdm(sorted(LOCAL_CONFIGS), desc="Configurations"):
    p_dl, p_sf = load_proba(DL_DIR, cfg), load_proba(SF_DIR, cfg)
    test_ids = LOCAL_CONFIGS[cfg]["test"]
    assert set(p_dl) == set(p_sf) == set(test_ids), f"Probabilities for c{cfg} do not match L_test"
    for img_id in test_ids:
        dl, sf, gt = to_full(p_dl[img_id]), to_full(p_sf[img_id]), GT[img_id]
        assert dl.shape[1:] == sf.shape[1:] == gt.shape, f"Size mismatch for {img_id}"
        valid = gt != IGNORE
        g = gt[valid].astype(np.int64)
        for w in WEIGHTS:
            pred = (w * sf + (1 - w) * dl).argmax(0)[valid]
            cm = np.bincount(g * 3 + pred, minlength=9)
            ens_rows.append({"config": cfg, "w": w, "id": img_id, "source": SOURCE_OF[img_id],
                             **dict(zip(CM_COLS, cm.tolist()))})
ens = pd.DataFrame(ens_rows)
ens.to_csv(OUT / "ensemble_cm_per_image.csv", index=False)

check = []
for w, arch in ((0.0, "DL"), (1.0, "SF")):
    orig = ev[(ev.arch == arch) & (ev.kind == "B") & (ev.seed == PROBA_SEED)]
    rec = ens[ens.w == w]
    check.append({"model": ARCH[arch],
                  "mIoU_litter original evaluation": compute(metric_arrays(orig))["mIoU_litter"],
                  "mIoU_litter from probabilities": compute(metric_arrays(rec))["mIoU_litter"]})
check = pd.DataFrame(check)
check["delta"] = check.iloc[:, 2] - check.iloc[:, 1]
display(check.round(5))
assert (check.delta.abs() < 0.005).all(), "Probabilities are inconsistent with the original evaluation"
print("Small differences come from rounding probabilities to uint8.")

### 5c. Weight selection on the partner configuration
For configuration c, the weight w with the highest mIoU litter on the partner configuration's test set is chosen and then scored on c. Ties go to the weight closest to 0.5.

In [ ]:
def cfg_score(cfg: int, w: float) -> float:
    return compute(metric_arrays(ens[(ens.config == cfg) & (ens.w == w)]))["mIoU_litter"]


grid = pd.DataFrame([{"config": c, "w": w, "mIoU_litter": cfg_score(c, w)}
                     for c in sorted(LOCAL_CONFIGS) for w in WEIGHTS])
grid_table = grid.pivot(index="config", columns="w", values="mIoU_litter")
print("mIoU litter per configuration and SegFormer weight (w):")
display(grid_table.round(4))

chosen = {}
for cfg, partner in PARTNER.items():
    scores = grid[grid.config == partner].set_index("w").mIoU_litter
    best = scores.max()
    candidates = [w for w in WEIGHTS if np.isclose(scores[w], best)]
    chosen[cfg] = min(candidates, key=lambda w: abs(w - 0.5))
print("Selected weights (from the partner configuration):", chosen)

selected = pd.concat([ens[(ens.config == c) & (ens.w == w)] for c, w in chosen.items()], ignore_index=True)
variants = {
    "DeepLab only": ens[ens.w == 0.0],
    "SegFormer only": ens[ens.w == 1.0],
    "Ensemble w=0.5": ens[ens.w == 0.5],
    "Ensemble selected w": selected,
}
summary_rows = []
for name, frame in variants.items():
    for grp, sub in [("all", frame)] + list(frame.groupby("source")):
        summary_rows.append({"variant": name, "group": grp, **compute(metric_arrays(per_image(sub)))})
ens_summary = pd.DataFrame(summary_rows)[["variant", "group"] + MAIN_METRICS]
ens_summary.to_csv(OUT / "ensemble_summary.csv", index=False)
display(ens_summary.round(4))

### 5d. Paired bootstrap: ensemble vs single model
Each image is represented by its confusion matrix averaged over the 3 configurations in which it is a test image (seed 44).

In [ ]:
img = {name: per_image(frame) for name, frame in variants.items()}
best_single = "SegFormer only" if compute(metric_arrays(img["SegFormer only"]))["mIoU_litter"] >= \
    compute(metric_arrays(img["DeepLab only"]))["mIoU_litter"] else "DeepLab only"
print("Best single model at seed 44:", best_single)

boot_ens = []
for ens_name in ("Ensemble selected w", "Ensemble w=0.5"):
    for single in ("DeepLab only", "SegFormer only"):
        res = paired_bootstrap(img[single], img[ens_name], "value_single", "value_ensemble")
        boot_ens.append(res.assign(ensemble=ens_name, baseline=single))
boot_ens = pd.concat(boot_ens, ignore_index=True)
boot_ens.to_csv(OUT / "significance_ensemble.csv", index=False)
show = boot_ens[boot_ens.group == "all"][["ensemble", "baseline", "metric", "value_single", "value_ensemble",
                                                "delta", "ci95_low", "ci95_high", "better"]]
print("Δ = ensemble − single model (all images):")
display(show.round(4))
print("Per-site results are in significance_ensemble.csv")

## 6. Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

ax = axes[0]
for kind, marker in (("A", "o"), ("B", "s")):
    for arch, color in (("DL", "tab:blue"), ("SF", "tab:orange")):
        sub = per_seed[(per_seed.model == kind) & (per_seed.arch == arch)]
        ax.errorbar(kind, sub.mIoU_litter.mean(), yerr=sub.mIoU_litter.std(ddof=1), fmt=marker, color=color,
                    capsize=4, label=f"{ARCH[arch]}" if kind == "A" else None,
                    transform=ax.transData)
ax.set_title("mIoU litter (mean ± std over seeds)")
ax.legend(fontsize=8)

ax = axes[1]
sub = boot_arch[(boot_arch.metric == "mIoU_litter")]
labels = [f"{m}\n{k}" for m, k in zip(sub.model, sub.group)]
ax.errorbar(range(len(sub)), sub.delta, yerr=[sub.delta - sub.ci95_low, sub.ci95_high - sub.delta],
            fmt="o", capsize=4)
ax.axhline(0, color="grey", lw=1)
ax.set_xticks(range(len(sub)), labels, fontsize=8)
ax.set_title("Δ mIoU litter, SegFormer − DeepLab (95% CI)")

ax = axes[2]
for cfg in sorted(LOCAL_CONFIGS):
    ax.plot(WEIGHTS, grid_table.loc[cfg], marker="o", lw=1, label=f"c{cfg}")
ax.set_xlabel("SegFormer weight (w)")
ax.set_title(f"Ensemble, seed {PROBA_SEED}: mIoU litter per configuration")
ax.legend(fontsize=7, ncol=2)

fig.tight_layout()
fig.savefig(OUT / "comparison_and_ensemble.png", dpi=150)
plt.show()

## 7. Final model choice
Decision rule:
1. Use the **ensemble** (both F models, weight = median of the selected weights) if, against the best single model, the selected-weight ensemble has significantly higher mIoU litter (95% CI above 0), a gain of at least `MIN_PRACTICAL_DIFF`, and total area MAE that is not significantly worse. The minimum gain matters because an ensemble runs two models, roughly doubling inference time.
2. Otherwise, use the F model of one architecture if its B model is significantly better (mIoU litter, bootstrap), not a tie under the seed-std rule, and at least `MIN_PRACTICAL_DIFF` better.
3. If neither holds, use the **faster** F model.

Note: the minimum-gain condition was added after the first run of this notebook. In that run the ensemble was chosen for a +0.0011 mIoU gain, which was statistically significant but not meaningful in practice. Report this change of rule openly.

In [ ]:
def row(df, **kw):
    sel = df
    for k, v in kw.items():
        sel = sel[sel[k] == v]
    assert len(sel) == 1, kw
    return sel.iloc[0]


ens_vs_best = row(boot_ens, ensemble="Ensemble selected w", baseline=best_single, group="all",
                  metric="mIoU_litter")
ens_mae = row(boot_ens, ensemble="Ensemble selected w", baseline=best_single, group="all",
              metric="area_MAE_total_pp")
arch_boot = row(boot_arch, model="B", group="all", metric="mIoU_litter")
arch_tie = row(tie_table, model="B", metric="mIoU_litter")
w_final = float(np.median(list(chosen.values())))
fast = "DL" if ms["DL"] <= ms["SF"] else "SF"

reasons = []
ensemble_wins = (ens_vs_best.ci95_low > 0 and ens_vs_best.delta >= MIN_PRACTICAL_DIFF
                 and not (ens_mae.ci95_low > 0))
arch_wins = (arch_boot.significant and arch_tie.verdict != "tie"
             and abs(arch_boot.delta) >= MIN_PRACTICAL_DIFF)
reasons.append(f"Ensemble (selected w) vs {best_single}: Δ mIoU litter {ens_vs_best.delta:+.4f} "
               f"(CI {ens_vs_best.ci95_low:+.4f} to {ens_vs_best.ci95_high:+.4f}); "
               f"Δ total area MAE {ens_mae.delta:+.3f} pp (CI {ens_mae.ci95_low:+.3f} to {ens_mae.ci95_high:+.3f}).")
reasons.append(f"Model B, SegFormer − DeepLab: Δ mIoU litter {arch_boot.delta:+.4f} "
               f"(CI {arch_boot.ci95_low:+.4f} to {arch_boot.ci95_high:+.4f}); seed-std rule: "
               f"{arch_tie.verdict}.")
reasons.append(f"Minimum practical difference: {MIN_PRACTICAL_DIFF:.2f} mIoU litter. Ensemble "
               f"{'meets' if ensemble_wins else 'does not meet'} the rule; architecture difference "
               f"{'meets' if arch_wins else 'does not meet'} the rule.")
reasons.append(f"Median inference time: DeepLab {ms['DL']:.1f} ms, SegFormer {ms['SF']:.1f} ms, "
               f"ensemble ≈{ms['DL'] + ms['SF']:.1f} ms per image (T4 GPU, short side 512).")

if ensemble_wins:
    decision = {"choice": "ensemble", "segformer_weight": w_final,
                "checkpoint": [str(DL_DIR / "deeplabv3plus_r101_final_local.pt"),
                               str(SF_DIR / "segformer_mit_b3_final_local.pt")]}
    text = (f"Use the ENSEMBLE: probability = {w_final:.2f} × SegFormer + {1 - w_final:.2f} × DeepLab, "
            "with both F models.")
elif arch_wins:
    winner = "SF" if arch_boot.delta > 0 else "DL"
    ckpt = "segformer_mit_b3_final_local.pt" if winner == "SF" else "deeplabv3plus_r101_final_local.pt"
    decision = {"choice": ARCH[winner], "checkpoint": [str((SF_DIR if winner == "SF" else DL_DIR) / ckpt)]}
    text = (f"Use the {ARCH[winner]} F model alone; the ensemble gives no meaningful gain "
            f"(< {MIN_PRACTICAL_DIFF:.2f} mIoU litter) over the best single model.")
else:
    ckpt = "segformer_mit_b3_final_local.pt" if fast == "SF" else "deeplabv3plus_r101_final_local.pt"
    decision = {"choice": ARCH[fast], "checkpoint": [str((SF_DIR if fast == "SF" else DL_DIR) / ckpt)]}
    text = f"No convincing difference; use the faster F model: {ARCH[fast]}."

for c in decision["checkpoint"]:
    if not Path(c).is_file():
        print("Warning: checkpoint not found in the inputs:", c)
decision.update({"min_practical_diff_mIoU": MIN_PRACTICAL_DIFF, "reasons": reasons, "recommendation": text, "selected_weight_per_config": chosen})
(OUT / "final_model_recommendation.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False))

print("RECOMMENDATION:", text)
for r in reasons:
    print(" -", r)

## 8. Output

In [ ]:
for f in sorted(OUT.iterdir()):
    print(f"{f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")